# triangle-barycentric — ex1: point-in-triangle test from (u, v)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `triangle-barycentric`. Running the final beacon cell reports progress against the `Geometry: Barycentric coords` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Barycentric coords` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`triangle-barycentric`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "triangle-barycentric"
DD_SUBTOPIC = "Geometry: Barycentric coords"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Triangle barycentric coordinates — quick refresher

Any point `P` in the plane of triangle `ABC` can be written

```
P = A + u * (B - A) + v * (C - A)
```

where `(u, v)` are the **barycentric coordinates** of `P` w.r.t. the edge basis `(B - A, C - A)`. `P` lies *inside* the triangle iff

```
u >= 0  AND  v >= 0  AND  u + v <= 1
```

(Equality on any bound puts `P` on an edge or vertex.) The third coordinate `w = 1 - u - v` is implied; together `(w, u, v)` are the conventional weights on `(A, B, C)`.

**Why ARENA's ray-triangle intersection cares.** Plugging the ray `R(s) = O + s * D` into the triangle's plane equation gives a 3x3 linear system whose solution is exactly `(s, u, v)`. Once you have it, the intersection test is the three inequalities above.

### Exercise 1 — point-in-triangle test from (u, v)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the barycentric inside-triangle predicate `u >= 0 & v >= 0 & u + v <= 1` to a batch of `(u, v)` coordinate pairs and return one boolean per point.
> Keywords: triangle, barycentric, predicate, visualization
> ```

**KCs targeted:** `barycentric-inside-predicate`, `barycentric-edge-basis`

Implement `ex1_inside_triangle(uvs)`. Given barycentric coordinates `uvs` of shape `(N, 2)` (column 0 is `u`, column 1 is `v`), return a boolean tensor of shape `(N,)` that is `True` exactly when the point `P = A + u*(B-A) + v*(C-A)` lies inside (or on the boundary of) the triangle `ABC`.

**Hint.** Three predicates ANDed together: `u >= 0`, `v >= 0`, `u + v <= 1`. No matrix solve, no projection — this drill is purely the inside test in barycentric space.

The visualization scatters the input points colored by inside/outside and overlays the canonical triangle `(0,0), (1,0), (0,1)` in `(u, v)` space so you can see the predicate boundary.

In [ ]:
def ex1_inside_triangle(uvs: Tensor) -> Tensor:
    u = uvs[:, 0]
    v = uvs[:, 1]
    return (u >= 0) & (v >= 0) & (u + v <= 1)


<details><summary>Solution</summary>

```python
def ex1_inside_triangle(uvs: Tensor) -> Tensor:
    u = uvs[:, 0]
    v = uvs[:, 1]
    return (u >= 0) & (v >= 0) & (u + v <= 1)
```

**Why three predicates, not four.** A 2-simplex has three edges (u=0, v=0, u+v=1). Each edge contributes one inequality; the interior is the intersection. The third coordinate `w = 1-u-v` is redundant — the constraint `w >= 0` is exactly `u + v <= 1`.

**Boundary handling.** Using `>=` and `<=` (not strict) makes edge/vertex points count as inside. This matches ARENA's `triangle_ray_intersects` convention; some renderers use strict inequalities to avoid double-counting shared edges.

**This drill skips the solve.** The full ray-triangle intersection (Möller-Trumbore) first solves a 3x3 system for `(s, u, v)`, then applies this predicate plus `s >= 0`. Splitting the predicate from the solve lets each subskill be drilled independently.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()